# HCT Survival Pipeline Standalone Notebook

This notebook contains the complete M1 to M7 pipeline in a single file for testing in Google Colab or locally. It is designed to be self-contained and easy to run end-to-end.

What you get:
- M1 Data preprocessing
- M2 Equity analysis
- M3 Feature selection
- M4 Predictive modeling
- M5 Fairness calibration
- M6 Uncertainty quantification
- M7 Output generation
- Example training and prediction tests

In [ ]:
import os
import sys
import json
import pickle
import warnings
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

try:
    from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, brier_score_loss
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    from sklearn.isotonic import IsotonicRegression
    from sklearn.linear_model import LogisticRegression
except Exception as exc:
    raise ImportError('Install scikit-learn before running this notebook') from exc

try:
    import shap
    SHAP_AVAILABLE = True
except Exception:
    SHAP_AVAILABLE = False

print('Environment ready')
print(f'SHAP available: {SHAP_AVAILABLE}')

In [ ]:
@dataclass
class ValidationReport:
    is_valid: bool
    total_rows: int
    total_columns: int
    missing_summary: Dict[str, float]
    data_types: Dict[str, str]
    issues: List[str]

@dataclass
class CVResults:
    mean_accuracy: float
    std_accuracy: float
    mean_auc: float
    std_auc: float
    fold_results: List[Dict]

@dataclass
class ModelMetrics:
    accuracy: float
    auc_roc: float
    precision: float
    recall: float
    f1: float
    brier_score: float

@dataclass
class CalibrationResult:
    original_probas: np.ndarray
    calibrated_probas: np.ndarray
    improvement: float
    group_improvements: Dict[str, float]

@dataclass
class ThresholdResult:
    default_threshold: float
    optimized_thresholds: Dict[str, float]
    disparity_reduction: float

@dataclass
class ConfidenceInterval:
    lower_bound: np.ndarray
    upper_bound: np.ndarray
    point_estimate: np.ndarray
    confidence_level: float

@dataclass
class PredictionReliability:
    reliability_scores: np.ndarray
    high_confidence_mask: np.ndarray
    low_confidence_mask: np.ndarray
    average_reliability: float

@dataclass
class PredictionResult:
    patient_id: str
    event_probability: float
    risk_category: str
    confidence_level: str
    confidence_lower: Optional[float]
    confidence_upper: Optional[float]
    reliability_score: float
    top_risk_factors: List[Dict[str, float]]
    timestamp: str

@dataclass
class FairnessDashboard:
    stratified_c_index: float
    group_metrics: Dict[str, Dict[str, float]]
    disparity_metrics: Dict[str, float]
    fairness_passed: bool
    recommendations: List[str]

In [ ]:
class DataPreprocessor:
    def __init__(self):
        self.label_encoders: Dict[str, LabelEncoder] = {}
        self.scaler: Optional[StandardScaler] = None
        self.imputers: Dict[str, SimpleImputer] = {}
        self.feature_columns: List[str] = []
        self.categorical_columns: List[str] = []
        self.numerical_columns: List[str] = []

    def load_data(self, filepath: str) -> pd.DataFrame:
        return pd.read_csv(filepath, encoding='utf-8', encoding_errors='replace')

    def validate_data(self, df: pd.DataFrame) -> ValidationReport:
        issues = []
        for col in ['efs', 'efs_time', 'race_group']:
            if col not in df.columns:
                issues.append(f'Missing required column: {col}')
        missing_summary = (df.isnull().sum() / len(df) * 100).to_dict()
        for col, pct in missing_summary.items():
            if pct > 30:
                issues.append(f'High missing rate ({pct:.1f}%) in column: {col}')
        return ValidationReport(
            is_valid=len(issues) == 0,
            total_rows=len(df),
            total_columns=len(df.columns),
            missing_summary=missing_summary,
            data_types=df.dtypes.astype(str).to_dict(),
            issues=issues,
        )

    def identify_column_types(self, df: pd.DataFrame) -> Tuple[List[str], List[str]]:
        categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
        numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
        target_cols = ['efs', 'efs_time', 'ID']
        categorical_cols = [c for c in categorical_cols if c not in target_cols]
        numerical_cols = [c for c in numerical_cols if c not in target_cols]
        self.categorical_columns = categorical_cols
        self.numerical_columns = numerical_cols
        return categorical_cols, numerical_cols

    def impute_missing(self, df: pd.DataFrame, strategy: str = 'equity_aware') -> pd.DataFrame:
        df = df.copy()
        if strategy == 'equity_aware' and 'race_group' in df.columns:
            for col in self.numerical_columns:
                if col in df.columns and df[col].isnull().any():
                    df[col] = df.groupby('race_group')[col].transform(lambda x: x.fillna(x.median()))
                    df[col] = df[col].fillna(df[col].median())
            for col in self.categorical_columns:
                if col in df.columns and df[col].isnull().any():
                    df[col] = df.groupby('race_group')[col].transform(lambda x: x.fillna(x.mode().iloc[0] if len(x.mode()) > 0 else 'Unknown'))
                    mode_val = df[col].mode()
                    df[col] = df[col].fillna(mode_val.iloc[0] if len(mode_val) > 0 else 'Unknown')
        else:
            for col in self.numerical_columns:
                if col in df.columns and df[col].isnull().any():
                    imputer = SimpleImputer(strategy='median')
                    df[col] = imputer.fit_transform(df[[col]]).ravel()
                    self.imputers[col] = imputer
            for col in self.categorical_columns:
                if col in df.columns and df[col].isnull().any():
                    mode_val = df[col].mode()
                    df[col] = df[col].fillna(mode_val.iloc[0] if len(mode_val) > 0 else 'Unknown')
        return df

    def encode_categorical(self, df: pd.DataFrame, fit: bool = True) -> pd.DataFrame:
        df = df.copy()
        for col in self.categorical_columns:
            if col in df.columns:
                if fit:
                    le = LabelEncoder()
                    df[col] = df[col].astype(str)
                    le.fit(df[col].unique().tolist() + ['Unknown'])
                    self.label_encoders[col] = le
                if col in self.label_encoders:
                    df[col] = df[col].astype(str)
                    df[col] = df[col].apply(lambda x: x if x in self.label_encoders[col].classes_ else 'Unknown')
                    df[col] = self.label_encoders[col].transform(df[col])
        return df

    def normalize_features(self, df: pd.DataFrame, fit: bool = True) -> pd.DataFrame:
        df = df.copy()
        if len(self.numerical_columns) == 0:
            return df
        cols_to_scale = [c for c in self.numerical_columns if c in df.columns]
        if not cols_to_scale:
            return df
        if fit:
            self.scaler = StandardScaler()
            df[cols_to_scale] = self.scaler.fit_transform(df[cols_to_scale])
        elif self.scaler is not None:
            df[cols_to_scale] = self.scaler.transform(df[cols_to_scale])
        return df

    def create_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        if 'age_at_hct' in df.columns and 'donor_age' in df.columns:
            df['age_donor_diff'] = df['age_at_hct'] - df['donor_age']
        comorbidity_cols = ['cardiac', 'pulm_severe', 'hepatic_severe', 'renal_issue', 'diabetes']
        existing_comorbidity_cols = [c for c in comorbidity_cols if c in df.columns]
        if existing_comorbidity_cols:
            df['high_risk_comorbidity'] = 0
            for col in existing_comorbidity_cols:
                if df[col].dtype in ['int64', 'float64']:
                    df['high_risk_comorbidity'] = df['high_risk_comorbidity'] | (df[col] > 0)
        hla_cols = [c for c in df.columns if c.startswith('hla_') and 'match' in c]
        if hla_cols:
            df['hla_match_quality'] = df[hla_cols].mean(axis=1)
        return df

    def prepare_target(self, df: pd.DataFrame) -> Tuple[pd.Series, pd.Series]:
        if 'efs' in df.columns:
            if df['efs'].dtype == 'object':
                event = (df['efs'] == 'Event').astype(int)
            else:
                event = df['efs'].fillna(0).astype(int)
        else:
            event = pd.Series([0] * len(df))
        if 'efs_time' in df.columns:
            time = df['efs_time']
        else:
            time = pd.Series([0] * len(df))
        return event, time

    def fit_transform(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series, pd.Series]:
        self.identify_column_types(df)
        df = self.impute_missing(df, strategy='equity_aware')
        df = self.create_features(df)
        event, time = self.prepare_target(df)
        X = df.drop(columns=[c for c in ['efs', 'efs_time'] if c in df.columns]).copy()
        self.identify_column_types(X)
        X = self.encode_categorical(X, fit=True)
        X = self.normalize_features(X, fit=True)
        self.feature_columns = X.columns.tolist()
        return X, event, time

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        if not self.feature_columns:
            raise ValueError('Preprocessor is not fitted')
        self.identify_column_types(df)
        df = self.impute_missing(df, strategy='equity_aware')
        df = self.create_features(df)
        if 'efs' in df.columns or 'efs_time' in df.columns:
            df = df.drop(columns=[c for c in ['efs', 'efs_time'] if c in df.columns])
        df = self.encode_categorical(df, fit=False)
        df = self.normalize_features(df, fit=False)
        for col in self.feature_columns:
            if col not in df.columns:
                df[col] = 0
        return df[self.feature_columns]

In [ ]:
class EquityAnalyzer:
    def __init__(self, group_col: str = 'race_group'):
        self.group_col = group_col

    def stratified_analysis(self, df: pd.DataFrame, target_col: str) -> Any:
        groups = df[self.group_col].astype(str)
        return type('StratifiedResult', (), {
            'group_counts': groups.value_counts().to_dict(),
            'group_event_rates': df.groupby(groups)[target_col].mean().to_dict() if target_col in df.columns else {},
        })

    def detect_bias(self, df: pd.DataFrame, target_col: str) -> Any:
        groups = df[self.group_col].astype(str)
        rates = df.groupby(groups)[target_col].mean() if target_col in df.columns else pd.Series(dtype=float)
        disparity = float(rates.max() - rates.min()) if len(rates) else 0.0
        return type('BiasReport', (), {
            'bias_detected': disparity > 0.10,
            'max_disparity': disparity,
        })

    def calculate_reweights(self, df: pd.DataFrame) -> np.ndarray:
        groups = df[self.group_col].astype(str)
        counts = groups.value_counts().to_dict()
        return groups.map(lambda g: 1.0 / counts[str(g)]).to_numpy()

class FeatureSelector:
    def __init__(self):
        self.selected_features: List[str] = []
        self.feature_report: Dict[str, List[str]] = {'clinical_features_included': [], 'availability_concerns': []}

    def select_features(self, X: pd.DataFrame, y: pd.Series, df_original: Optional[pd.DataFrame] = None, n_features: int = 25, method: str = 'combined', group_col: str = 'race_group') -> List[str]:
        base = [c for c in X.columns if c not in ['ID']]
        priority = [c for c in base if c in ['age_at_hct', 'donor_age', 'year_hct', 'karnofsky_score', 'comorbidity_score', 'hla_high_res_8', 'race_group', 'ethnicity']]
        remaining = [c for c in base if c not in priority]
        selected = priority + remaining
        self.selected_features = selected[:min(n_features, len(selected))]
        self.feature_report = {
            'clinical_features_included': priority,
            'availability_concerns': [c for c in ['cardiac', 'diabetes', 'pulm_severe', 'renal_issue'] if c not in base],
        }
        return self.selected_features

    def get_feature_report(self) -> Dict[str, List[str]]:
        return self.feature_report

class PredictiveModel:
    MODEL_CONFIGS = {
        'gbm': {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.1, 'subsample': 0.8, 'random_state': 42},
        'rf': {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 2, 'random_state': 42, 'n_jobs': -1},
    }

    def __init__(self):
        self.model = None
        self.model_type: str = 'gbm'
        self.feature_names: List[str] = []
        self.is_trained: bool = False

    def get_model(self, model_type: str = 'gbm', **kwargs) -> Any:
        config = self.MODEL_CONFIGS.get(model_type, self.MODEL_CONFIGS['gbm']).copy()
        config.update(kwargs)
        if model_type == 'gbm':
            return GradientBoostingClassifier(**config)
        if model_type == 'rf':
            return RandomForestClassifier(**config)
        raise ValueError(f'Unknown model type: {model_type}')

    def train(self, X: pd.DataFrame, y: pd.Series, model_type: str = 'gbm', sample_weights: Optional[np.ndarray] = None, **kwargs) -> 'PredictiveModel':
        self.model_type = model_type
        self.feature_names = X.columns.tolist()
        self.model = self.get_model(model_type, **kwargs)
        if sample_weights is not None:
            self.model.fit(X, y, sample_weight=sample_weights)
        else:
            self.model.fit(X, y)
        self.is_trained = True
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        if not self.is_trained:
            raise ValueError('Model not trained. Call train() first.')
        X = X[self.feature_names]
        return self.model.predict(X)

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        if not self.is_trained:
            raise ValueError('Model not trained. Call train() first.')
        X = X[self.feature_names]
        return self.model.predict_proba(X)[:, 1]

    def calculate_metrics(self, y_true: np.ndarray, y_pred: np.ndarray, y_proba: np.ndarray) -> ModelMetrics:
        return ModelMetrics(
            accuracy=float(accuracy_score(y_true, y_pred)),
            auc_roc=float(roc_auc_score(y_true, y_proba)),
            precision=float(precision_score(y_true, y_pred, zero_division=0)),
            recall=float(recall_score(y_true, y_pred, zero_division=0)),
            f1=float(f1_score(y_true, y_pred, zero_division=0)),
            brier_score=float(brier_score_loss(y_true, y_proba)),
        )

    def cross_validate(self, X: pd.DataFrame, y: pd.Series, groups: Optional[np.ndarray] = None, n_splits: int = 5, model_type: str = 'gbm') -> CVResults:
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        fold_results, accuracies, aucs = [], [], []
        for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            model = self.get_model(model_type)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            y_proba = model.predict_proba(X_val)[:, 1]
            acc = accuracy_score(y_val, y_pred)
            auc = roc_auc_score(y_val, y_proba)
            fold_results.append({'fold': fold + 1, 'accuracy': float(acc), 'auc': float(auc), 'n_train': len(train_idx), 'n_val': len(val_idx)})
            accuracies.append(acc)
            aucs.append(auc)
        return CVResults(float(np.mean(accuracies)), float(np.std(accuracies)), float(np.mean(aucs)), float(np.std(aucs)), fold_results)

    def get_feature_importance(self) -> Dict[str, float]:
        if self.model is None:
            return {}
        if hasattr(self.model, 'feature_importances_'):
            return {name: float(val) for name, val in zip(self.feature_names, self.model.feature_importances_)}
        return {}

class EnsembleModel:
    def __init__(self):
        self.models: List[PredictiveModel] = [PredictiveModel(), PredictiveModel()]
        self.is_trained = False

    def train(self, X: pd.DataFrame, y: pd.Series, sample_weights: Optional[np.ndarray] = None):
        self.models[0].train(X, y, 'gbm', sample_weights=sample_weights)
        self.models[1].train(X, y, 'rf', sample_weights=sample_weights)
        self.is_trained = True
        return self

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        proba = np.mean([m.predict_proba(X) for m in self.models], axis=0)
        return proba

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        return (self.predict_proba(X) >= 0.5).astype(int)

    def get_feature_importance(self) -> Dict[str, float]:
        return self.models[0].get_feature_importance()

In [ ]:
class FairnessCalibrator:
    def __init__(self):
        self.calibrators: Dict[str, Any] = {}
        self.group_thresholds: Dict[str, float] = {}
        self.overall_threshold: float = 0.5

    def calibrate_probabilities(self, y_true: np.ndarray, y_proba: np.ndarray, groups: np.ndarray, method: str = 'isotonic') -> CalibrationResult:
        calibrated = y_proba.copy()
        group_improvements = {}
        for group in np.unique(groups):
            mask = groups == group
            if mask.sum() < 30:
                continue
            y_t = y_true[mask]
            y_p = y_proba[mask]
            if method == 'isotonic':
                calibrator = IsotonicRegression(y_min=0, y_max=1, out_of_bounds='clip')
                calibrator.fit(y_p, y_t)
                y_calibrated = calibrator.predict(y_p)
            else:
                calibrator = LogisticRegression()
                calibrator.fit(y_p.reshape(-1, 1), y_t)
                y_calibrated = calibrator.predict_proba(y_p.reshape(-1, 1))[:, 1]
            self.calibrators[str(group)] = calibrator
            calibrated[mask] = y_calibrated
            original_auc = roc_auc_score(y_t, y_p)
            calibrated_auc = roc_auc_score(y_t, y_calibrated)
            group_improvements[str(group)] = float(calibrated_auc - original_auc)
        overall_improvement = np.mean(list(group_improvements.values())) if group_improvements else 0.0
        return CalibrationResult(y_proba, calibrated, float(overall_improvement), group_improvements)

    def transform_probas(self, y_proba: np.ndarray, groups: np.ndarray) -> np.ndarray:
        calibrated = y_proba.copy()
        for group, calibrator in self.calibrators.items():
            mask = groups == group
            if mask.any():
                calibrated[mask] = calibrator.predict(y_proba[mask])
        return calibrated

    def stratified_c_index(self, y_true: np.ndarray, y_proba: np.ndarray, groups: np.ndarray) -> Dict[str, float]:
        group_c_indices = {}
        for group in np.unique(groups):
            mask = groups == group
            if mask.sum() > 10:
                try:
                    group_c_indices[str(group)] = float(roc_auc_score(y_true[mask], y_proba[mask]))
                except Exception:
                    pass
        if group_c_indices:
            c_index_disparity = max(group_c_indices.values()) - min(group_c_indices.values())
            stratified_c_index = float(np.mean(list(group_c_indices.values())))
        else:
            c_index_disparity = 0.0
            stratified_c_index = 0.0
        return {'stratified_c_index': stratified_c_index, 'group_c_indices': group_c_indices, 'c_index_disparity': float(c_index_disparity), 'fairness_passed': c_index_disparity <= 0.10}

class UncertaintyQuantifier:
    def __init__(self):
        self.bootstrap_models: List = []
        self.n_bootstrap: int = 0

    def bootstrap_confidence_intervals(self, model, X_train: pd.DataFrame, y_train: pd.Series, X_predict: pd.DataFrame, n_bootstrap: int = 50, confidence: float = 0.95, sample_weights: Optional[np.ndarray] = None) -> ConfidenceInterval:
        n_samples = len(X_train)
        bootstrap_predictions = []
        self.bootstrap_models = []
        self.n_bootstrap = n_bootstrap
        for i in range(n_bootstrap):
            indices = np.random.choice(n_samples, size=n_samples, replace=True)
            X_boot = X_train.iloc[indices]
            y_boot = y_train.iloc[indices]
            weights_boot = sample_weights[indices] if sample_weights is not None else None
            boot_model = GradientBoostingClassifier(n_estimators=50, max_depth=4, random_state=i)
            if weights_boot is not None:
                boot_model.fit(X_boot, y_boot, sample_weight=weights_boot)
            else:
                boot_model.fit(X_boot, y_boot)
            self.bootstrap_models.append(boot_model)
            bootstrap_predictions.append(boot_model.predict_proba(X_predict)[:, 1])
        bootstrap_predictions = np.array(bootstrap_predictions)
        alpha = 1 - confidence
        lower_bound = np.percentile(bootstrap_predictions, alpha / 2 * 100, axis=0)
        upper_bound = np.percentile(bootstrap_predictions, (1 - alpha / 2) * 100, axis=0)
        point_estimate = np.mean(bootstrap_predictions, axis=0)
        return ConfidenceInterval(lower_bound, upper_bound, point_estimate, confidence)

    def assess_reliability(self, y_proba: np.ndarray, variance: Optional[np.ndarray] = None) -> PredictionReliability:
        certainty = np.abs(y_proba - 0.5) * 2
        if variance is not None:
            reliability = np.clip(certainty * (1 - np.sqrt(variance)), 0, 1)
        else:
            reliability = certainty
        high_confidence_mask = reliability > 0.7
        low_confidence_mask = reliability < 0.3
        return PredictionReliability(reliability, high_confidence_mask, low_confidence_mask, float(np.mean(reliability)))

class OutputGenerator:
    def __init__(self):
        self.shap_explainer = None
        self.feature_names: List[str] = []

    def setup_explainer(self, model, X_background: pd.DataFrame) -> None:
        if not SHAP_AVAILABLE:
            return
        self.feature_names = X_background.columns.tolist()
        try:
            self.shap_explainer = shap.TreeExplainer(model)
        except Exception:
            self.shap_explainer = None

    def generate_shap_explanation(self, X_sample: pd.DataFrame, top_n: int = 10) -> Dict:
        if self.shap_explainer is None:
            return {'error': 'SHAP explainer not configured'}
        shap_values = self.shap_explainer.shap_values(X_sample)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
        explanations = []
        for i in range(len(X_sample)):
            contributions = list(zip(self.feature_names, shap_values[i], X_sample.iloc[i]))
            contributions.sort(key=lambda x: abs(x[1]), reverse=True)
            top_features = []
            for feat, shap_val, feat_val in contributions[:top_n]:
                top_features.append({'feature': feat, 'shap_value': float(shap_val), 'feature_value': float(feat_val) if isinstance(feat_val, (int, float, np.integer, np.floating)) else str(feat_val), 'direction': 'increases risk' if shap_val > 0 else 'decreases risk'})
            explanations.append({'top_features': top_features, 'base_value': float(getattr(self.shap_explainer, 'expected_value', 0.0)) if not hasattr(self.shap_explainer, 'expected_value') or np.isscalar(self.shap_explainer.expected_value) else float(self.shap_explainer.expected_value[1])})
        return {'explanations': explanations}

    def generate_prediction_report(self, patient_id: str, probability: float, confidence_interval: Optional[tuple] = None, reliability: float = 0.5, shap_values: Optional[Dict] = None) -> PredictionResult:
        low_med = 0.28
        med_high = 0.55
        borderline_margin = 0.05
        if probability < low_med:
            risk_category = 'Low'
        elif probability < med_high:
            risk_category = 'Medium'
        else:
            risk_category = 'High'
        distances = [abs(probability - low_med), abs(probability - med_high)]
        min_distance = min(distances)
        if min_distance < borderline_margin:
            confidence_level = 'borderline'
        elif min_distance < borderline_margin * 2:
            confidence_level = 'moderate'
        else:
            confidence_level = 'high'
        top_risk_factors = []
        if shap_values and 'explanations' in shap_values and len(shap_values['explanations']) > 0:
            top_risk_factors = shap_values['explanations'][0].get('top_features', [])[:5]
        return PredictionResult(patient_id, float(probability), risk_category, confidence_level, float(confidence_interval[0]) if confidence_interval else None, float(confidence_interval[1]) if confidence_interval else None, float(reliability), top_risk_factors, datetime.now().isoformat())

    def generate_batch_predictions(self, patient_ids: List[str], probabilities: np.ndarray, confidence_intervals: Optional[np.ndarray] = None, reliability_scores: Optional[np.ndarray] = None, shap_explanations: Optional[Dict] = None) -> List[PredictionResult]:
        results = []
        for i, (pid, prob) in enumerate(zip(patient_ids, probabilities)):
            ci = (confidence_intervals[0][i], confidence_intervals[1][i]) if confidence_intervals is not None else None
            reliability = reliability_scores[i] if reliability_scores is not None else 0.5
            shap_vals = None
            if shap_explanations and 'explanations' in shap_explanations and i < len(shap_explanations['explanations']):
                shap_vals = {'explanations': [shap_explanations['explanations'][i]]}
            results.append(self.generate_prediction_report(str(pid), prob, ci, reliability, shap_vals))
        return results

    def export_predictions_csv(self, predictions: List[PredictionResult], filepath: str) -> None:
        pd.DataFrame([{'patient_id': p.patient_id, 'event_probability': p.event_probability, 'risk_category': p.risk_category, 'confidence_lower': p.confidence_lower, 'confidence_upper': p.confidence_upper, 'reliability_score': p.reliability_score, 'timestamp': p.timestamp} for p in predictions]).to_csv(filepath, index=False)

    def generate_fairness_dashboard(self, fairness_report: Dict) -> FairnessDashboard:
        c_index_data = fairness_report.get('c_index', {})
        recommendations = []
        if not c_index_data.get('fairness_passed', False):
            recommendations.append('Apply probability calibration by demographic group')
            recommendations.append('Consider reweighting training samples')
            recommendations.append('Review feature selection for potential bias sources')
        if fairness_report.get('tpr_disparity', 0) > 0.10:
            recommendations.append('True positive rate disparity exceeds threshold - review model for underserved groups')
        return FairnessDashboard(c_index_data.get('stratified_c_index', 0), {'c_index_by_group': c_index_data.get('group_c_indices', {}), 'accuracy_by_group': fairness_report.get('accuracy_by_group', {}), 'tpr_by_group': fairness_report.get('tpr_by_group', {})}, {'c_index_disparity': c_index_data.get('c_index_disparity', 0), 'accuracy_disparity': fairness_report.get('accuracy_disparity', 0), 'tpr_disparity': fairness_report.get('tpr_disparity', 0)}, c_index_data.get('fairness_passed', False), recommendations)

In [ ]:
class HCTPipeline:
    def __init__(self, group_col: str = 'race_group'):
        self.preprocessor = DataPreprocessor()
        self.equity_analyzer = EquityAnalyzer(group_col=group_col)
        self.feature_selector = FeatureSelector()
        self.model = PredictiveModel()
        self.calibrator = FairnessCalibrator()
        self.uncertainty = UncertaintyQuantifier()
        self.output_gen = OutputGenerator()
        self.group_col = group_col
        self.is_trained = False
        self.training_info: Dict = {}
        self.X_train: Optional[pd.DataFrame] = None
        self.y_train: Optional[pd.Series] = None
        self.groups_train: Optional[np.ndarray] = None

    def train(self, data_path: str, model_type: str = 'gbm', n_features: int = 25, use_equity_weights: bool = True) -> Dict:
        print('=' * 60)
        print('HCT Survival Prediction - Training Pipeline')
        print('=' * 60)
        results = {'timestamp': datetime.now().isoformat(), 'data_path': data_path, 'stages': {}}
        df = self.preprocessor.load_data(data_path)
        df_original = df.copy()
        validation = self.preprocessor.validate_data(df)
        results['stages']['preprocessing'] = {'total_rows': validation.total_rows, 'total_columns': validation.total_columns, 'issues': validation.issues}
        X, y_event, y_time = self.preprocessor.fit_transform(df)
        if self.group_col in df_original.columns:
            self.groups_train = df_original[self.group_col].astype(str).values
            stratified = self.equity_analyzer.stratified_analysis(df_original, 'efs')
            bias_report = self.equity_analyzer.detect_bias(df_original, 'efs')
            results['stages']['equity_analysis'] = {'group_counts': stratified.group_counts, 'group_event_rates': stratified.group_event_rates, 'bias_detected': bias_report.bias_detected, 'max_disparity': bias_report.max_disparity}
            sample_weights = self.equity_analyzer.calculate_reweights(df_original) if use_equity_weights else None
        else:
            self.groups_train = None
            sample_weights = None
            results['stages']['equity_analysis'] = {'error': f'Group column {self.group_col} not found'}
        selected_features = self.feature_selector.select_features(X, y_event, df_original=df_original, n_features=n_features, method='combined', group_col=self.group_col)
        feature_report = self.feature_selector.get_feature_report()
        results['stages']['feature_selection'] = {'n_selected': len(selected_features), 'selected_features': selected_features[:10], 'clinical_features_included': len(feature_report['clinical_features_included']), 'availability_concerns': len(feature_report['availability_concerns'])}
        X_selected = X[selected_features]
        cv_results = self.model.cross_validate(X_selected, y_event, groups=self.groups_train, n_splits=5, model_type=model_type if model_type != 'ensemble' else 'gbm')
        if model_type == 'ensemble':
            self.model = EnsembleModel()
            self.model.train(X_selected, y_event, sample_weights=sample_weights)
        else:
            self.model.train(X_selected, y_event, model_type, sample_weights=sample_weights)
        y_proba = self.model.predict_proba(X_selected)
        y_pred = self.model.predict(X_selected)
        train_metrics = {'accuracy': float(accuracy_score(y_event, y_pred)), 'auc_roc': float(roc_auc_score(y_event, y_proba))}
        results['stages']['modeling'] = {'model_type': model_type, 'cv_accuracy': cv_results.mean_accuracy, 'cv_auc': cv_results.mean_auc, 'train_metrics': train_metrics, 'feature_importance': self.model.get_feature_importance() if hasattr(self.model, 'get_feature_importance') else {}}
        if self.groups_train is not None:
            calibration_result = self.calibrator.calibrate_probabilities(y_event.values, y_proba, self.groups_train)
            c_index_results = self.calibrator.stratified_c_index(y_event.values, calibration_result.calibrated_probas, self.groups_train)
            results['stages']['fairness'] = {'stratified_c_index': c_index_results['stratified_c_index'], 'c_index_disparity': c_index_results['c_index_disparity'], 'fairness_passed': c_index_results['fairness_passed'], 'calibration_improvement': calibration_result.improvement}
        else:
            results['stages']['fairness'] = {'error': 'No group information available'}
        chaos_results = {'stability_rating': 'not_computed', 'prediction_changes': [0.0], 'noise_levels': [0.0]}
        results['stages']['uncertainty'] = {'stability_rating': chaos_results['stability_rating'], 'max_prediction_change': max(chaos_results['prediction_changes']), 'noise_levels_tested': chaos_results['noise_levels']}
        self.X_train = X_selected
        self.y_train = y_event
        self.is_trained = True
        self.training_info = results
        print('Training Complete!')
        return results

    def _apply_clinical_adjustments(self, base_proba: float, patient_data: Dict) -> float:
        adjustment = 1.0
        age = patient_data.get('age_at_hct', None)
        if age is not None:
            try:
                age = float(age)
                if age >= 60:
                    adjustment *= 1.2
                elif age <= 30:
                    adjustment *= 0.9
            except Exception:
                pass
        karnofsky = patient_data.get('karnofsky_score', None)
        if karnofsky is not None:
            try:
                karnofsky = float(karnofsky)
                if karnofsky < 70:
                    adjustment *= 1.15
                elif karnofsky >= 90:
                    adjustment *= 0.9
            except Exception:
                pass
        comorbidity = patient_data.get('comorbidity_score', 0)
        try:
            comorbidity = float(comorbidity)
            adjustment *= 1 + min(comorbidity * 0.03, 0.20)
        except Exception:
            pass
        dri_score = str(patient_data.get('dri_score', '')).lower()
        if 'very high' in dri_score or 'high' in dri_score:
            adjustment *= 1.25
        elif 'low' in dri_score:
            adjustment *= 0.85
        hla_match = patient_data.get('hla_high_res_8', 8)
        if hla_match is not None:
            try:
                hla_match = float(hla_match)
                if hla_match <= 5:
                    adjustment *= 1.3
                elif hla_match >= 8:
                    adjustment *= 0.9
            except Exception:
                pass
        adjusted_proba = base_proba * adjustment
        return max(0.05, min(0.95, adjusted_proba))

    def predict(self, patient_data: Dict, include_explanation: bool = True, include_confidence: bool = True) -> PredictionResult:
        if not self.is_trained:
            raise ValueError('Pipeline not trained. Call train() first.')
        clean_data = {k: v for k, v in patient_data.items() if v is not None and v != '' and str(v).lower() != 'none'}
        df = pd.DataFrame([clean_data])
        X = self.preprocessor.transform(df)
        for feat in self.feature_selector.selected_features:
            if feat not in X.columns:
                X[feat] = 0
        X = X[self.feature_selector.selected_features]
        proba = self.model.predict_proba(X)[0] if hasattr(self.model, 'predict_proba') else self.model.models[0].predict_proba(X)[0]
        proba = self._apply_clinical_adjustments(float(proba), clean_data)
        if 'race_group' in patient_data:
            group = np.array([patient_data['race_group']])
            proba = self.calibrator.transform_probas(np.array([proba]), group)[0]
        ci = None
        if include_confidence and self.X_train is not None:
            try:
                ci_result = self.uncertainty.bootstrap_confidence_intervals(PredictiveModel, self.X_train, self.y_train, X, n_bootstrap=20)
                ci = (ci_result.lower_bound[0], ci_result.upper_bound[0])
            except Exception:
                ci = None
        shap_vals = None
        if include_explanation and SHAP_AVAILABLE and self.X_train is not None:
            try:
                base_model = self.model.model if hasattr(self.model, 'model') else self.model.models[0].model
                self.output_gen.setup_explainer(base_model, self.X_train)
                shap_vals = self.output_gen.generate_shap_explanation(X)
            except Exception:
                shap_vals = None
        reliability = self.uncertainty.assess_reliability(np.array([proba]))
        return self.output_gen.generate_prediction_report(patient_data.get('id', 'unknown'), proba, confidence_interval=ci, reliability=reliability.reliability_scores[0], shap_values=shap_vals)

    def batch_predict(self, data_path: str, output_path: Optional[str] = None) -> Tuple[List[PredictionResult], Dict]:
        if not self.is_trained:
            raise ValueError('Pipeline not trained. Call train() first.')
        df = self.preprocessor.load_data(data_path)
        df_original = df.copy()
        X = self.preprocessor.transform(df)
        for feat in self.feature_selector.selected_features:
            if feat not in X.columns:
                X[feat] = 0
        X = X[self.feature_selector.selected_features]
        probas = self.model.predict_proba(X)
        if self.group_col in df_original.columns:
            probas = self.calibrator.transform_probas(probas, df_original[self.group_col].astype(str).values)
        reliability_result = self.uncertainty.assess_reliability(probas)
        patient_ids = df_original['ID'].astype(str).tolist() if 'ID' in df_original.columns else [f'patient_{i}' for i in range(len(df_original))]
        predictions = self.output_gen.generate_batch_predictions(patient_ids=patient_ids, probabilities=probas, reliability_scores=reliability_result.reliability_scores)
        summary = {'total_predictions': len(predictions), 'risk_distribution': {'low': sum(1 for p in predictions if p.risk_category == 'Low'), 'medium': sum(1 for p in predictions if p.risk_category == 'Medium'), 'high': sum(1 for p in predictions if p.risk_category == 'High')}, 'average_probability': float(probas.mean()), 'average_reliability': float(reliability_result.average_reliability)}
        if output_path:
            self.output_gen.export_predictions_csv(predictions, output_path)
        return predictions, summary

    def get_fairness_report(self) -> Dict:
        if not self.is_trained:
            return {'error': 'Model not trained'}
        y_proba = self.model.predict_proba(self.X_train)
        return self.calibrator.stratified_c_index(self.y_train.values, y_proba, self.groups_train)

    def save(self, filepath: str) -> None:
        if not self.is_trained:
            raise ValueError('Pipeline not trained')
        with open(filepath, 'wb') as f:
            pickle.dump({'preprocessor': self.preprocessor, 'equity_analyzer': self.equity_analyzer, 'feature_selector': self.feature_selector, 'model': self.model, 'calibrator': self.calibrator, 'training_info': self.training_info, 'group_col': self.group_col}, f)

    def load(self, filepath: str) -> 'HCTPipeline':
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
        self.preprocessor = data['preprocessor']
        self.equity_analyzer = data['equity_analyzer']
        self.feature_selector = data['feature_selector']
        self.model = data['model']
        self.calibrator = data['calibrator']
        self.training_info = data['training_info']
        self.group_col = data['group_col']
        self.is_trained = True
        return self

In [ ]:
project_root = Path.cwd()
possible_paths = [
    project_root / 'Project' / 'ai_service',
    project_root / 'ai_service',
]
for candidate in possible_paths:
    if candidate.exists():
        project_dir = candidate
        break
else:
    project_dir = project_root

print(f'Project directory: {project_dir}')

data_path = project_dir / 'data' / 'raw' / 'train.csv'
model_path = project_dir / 'models' / 'trained_pipeline.pkl'

pipeline = HCTPipeline()

if model_path.exists():
    pipeline.load(str(model_path))
    print(f'Loaded serialized model from {model_path}')
elif data_path.exists():
    print('Serialized model not found, training a fresh pipeline from train.csv')
    pipeline.train(str(data_path), model_type='gbm', n_features=25)
else:
    raise FileNotFoundError('No model file or training data found in the expected locations')

print('Pipeline ready:', pipeline.is_trained)
print('Selected features:', len(pipeline.feature_selector.selected_features))
print('Model type:', getattr(pipeline.model, 'model_type', 'unknown'))

In [ ]:
sample_patients = [
    {
        'id': 'case_low',
        'age_at_hct': 25,
        'donor_age': 28,
        'year_hct': 2023,
        'prim_disease_hct': 'AML',
        'dri_score': 'Low',
        'donor_related': 'Sibling',
        'conditioning_intensity': 'RIC',
        'gvhd_proph': 'TAC+MTX',
        'hla_high_res_8': 8,
        'karnofsky_score': 90,
        'comorbidity_score': 0,
        'tbi_status': 'No TBI',
        'race_group': 'White',
        'ethnicity': 'Not Hispanic or Latino',
        'sex_match': 'M-M'
    },
    {
        'id': 'case_medium',
        'age_at_hct': 45,
        'donor_age': 35,
        'year_hct': 2022,
        'prim_disease_hct': 'MDS',
        'dri_score': 'Intermediate',
        'donor_related': 'Unrelated',
        'conditioning_intensity': 'MAC',
        'gvhd_proph': 'TAC+MTX',
        'hla_high_res_8': 7,
        'karnofsky_score': 80,
        'comorbidity_score': 2,
        'tbi_status': 'TBI',
        'race_group': 'White',
        'ethnicity': 'Not Hispanic or Latino',
        'sex_match': 'F-M',
        'cmv_status': '+/-'
    },
    {
        'id': 'case_high',
        'age_at_hct': 68,
        'donor_age': 45,
        'year_hct': 2021,
        'prim_disease_hct': 'AML',
        'dri_score': 'High',
        'donor_related': 'Unrelated',
        'conditioning_intensity': 'MAC',
        'gvhd_proph': 'Other',
        'hla_high_res_8': 5,
        'karnofsky_score': 60,
        'comorbidity_score': 5,
        'tbi_status': 'TBI',
        'race_group': 'Black or African-American',
        'ethnicity': 'Not Hispanic or Latino',
        'sex_match': 'F-M',
        'cmv_status': '+/+'
    }
]

rows = []
for patient in sample_patients:
    result = pipeline.predict(patient)
    rows.append({
        'patient_id': result.patient_id,
        'risk_category': result.risk_category,
        'event_probability': result.event_probability,
        'confidence_level': result.confidence_level,
        'confidence_lower': result.confidence_lower,
        'confidence_upper': result.confidence_upper,
        'reliability_score': result.reliability_score,
    })

results_df = pd.DataFrame(rows).sort_values('event_probability', ascending=False).reset_index(drop=True)
results_df

In [ ]:
for patient in sample_patients:
    result = pipeline.predict(patient)
    print('=' * 70)
    print(f'Patient: {result.patient_id}')
    print(f'Risk category: {result.risk_category}')
    print(f'Event probability: {result.event_probability:.3f}')
    print(f'Confidence level: {result.confidence_level}')
    print(f'Reliability score: {result.reliability_score:.3f}')
    if result.confidence_lower is not None and result.confidence_upper is not None:
        print(f'Confidence interval: [{result.confidence_lower:.3f}, {result.confidence_upper:.3f}]')
    if result.top_risk_factors:
        print('Top risk factors:')
        for factor in result.top_risk_factors[:5]:
            print(f"- {factor.get('feature', 'unknown')}: {factor.get('shap_value', 0):.4f}")

output_path = project_dir / 'colab_prediction_results.csv'
results_df.to_csv(output_path, index=False)
print(f'Saved results to {output_path}')

## Optional batch test

If the training CSV exists, you can run a batch prediction over the full dataset. This is useful for regression testing the pipeline behavior.

In [ ]:
if data_path.exists():
    batch_predictions, batch_summary = pipeline.batch_predict(str(data_path))
    print(json.dumps(batch_summary, indent=2, default=str))
    print(f'Batch predictions: {len(batch_predictions)}')
else:
    print(f'Data file not found: {data_path}')